# Lab3.5 — Benchmark DMA: latență și throughput

Ultimul task al Lab3 transformă experimentul DMA într-un benchmark repetabil. Scopul este separarea comportamentului de **latență** de comportamentul de **throughput**, două metrici esențiale în Edge Computing.


## Obiective

- efectuarea mai multor rulări pentru fiecare dimensiune;
- calculul mediei, medianei și deviației standard;
- estimarea latenței pentru transferuri mici;
- calculul throughput-ului pentru transferuri mari;
- reprezentarea grafică și interpretarea rezultatelor.


## 1. Inițializare


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

BITSTREAM_DIR = Path("bitstream")
BITSTREAM_DIR.mkdir(exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/onigas/PYNQ_Workshop/master/Session_4/bitstream"
FILES = ("dma_tutorial.bit", "dma_tutorial.hwh")

for filename in FILES:
    local_path = BITSTREAM_DIR / filename
    if not local_path.exists():
        print(f"Downloading {filename} ...")
        try:
            urlretrieve(f"{BASE_URL}/{filename}", local_path)
        except Exception as e:
            raise RuntimeError(
                f"Nu s-a putut descărca {filename}. "
                "Copiați manual fișierele dma_tutorial.bit și dma_tutorial.hwh "
                "în directorul Lab3/bitstream."
            ) from e

print("DMA overlay files are ready.")

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time
import statistics
import matplotlib.pyplot as plt

overlay = Overlay("bitstream/dma_tutorial.bit")
dma_send = overlay.axi_dma_from_ps_to_pl
dma_recv = overlay.axi_dma_from_pl_to_ps

## 2. Funcția de benchmark

Bufferele sunt alocate o singură dată pentru fiecare dimensiune, iar apoi transferul este repetat. Astfel măsurăm tranzacția DMA, nu costul alocării memoriei.


In [ ]:
def benchmark_dma(n_elements, repeats=10):
    inp = allocate(shape=(n_elements,), dtype=np.uint32)
    out = allocate(shape=(n_elements,), dtype=np.uint32)

    inp[:] = np.arange(n_elements, dtype=np.uint32)
    out[:] = 0

    times = []

    for _ in range(repeats):
        if hasattr(inp, "flush"):
            inp.flush()

        t0 = time.perf_counter()
        dma_recv.recvchannel.transfer(out)
        dma_send.sendchannel.transfer(inp)
        dma_send.sendchannel.wait()
        dma_recv.recvchannel.wait()
        t1 = time.perf_counter()

        if hasattr(out, "invalidate"):
            out.invalidate()

        if not np.array_equal(inp, out):
            del inp, out
            raise RuntimeError("DMA data verification failed")

        times.append(t1 - t0)

    nbytes = inp.nbytes
    del inp, out

    return {
        "elements": n_elements,
        "bytes": nbytes,
        "times": times,
        "mean_s": statistics.mean(times),
        "median_s": statistics.median(times),
        "stdev_s": statistics.stdev(times) if len(times) > 1 else 0.0,
    }

## 3. Rularea benchmark-ului


In [ ]:
sizes = (16, 64, 256, 1024, 4096, 16384, 65536)
REPEATS = 10

results = [benchmark_dma(n, REPEATS) for n in sizes]

print(f"{'Elements':>9s} {'Bytes':>10s} {'Mean ms':>10s} {'Std us':>10s} {'MB/s':>10s}")
print("-" * 56)

for r in results:
    throughput = r['bytes'] / r['mean_s'] / 1e6
    print(
        f"{r['elements']:9d} {r['bytes']:10d} "
        f"{r['mean_s']*1e3:10.3f} {r['stdev_s']*1e6:10.3f} "
        f"{throughput:10.3f}"
    )

## 4. Reprezentarea grafică


In [ ]:
bytes_list = [r['bytes'] for r in results]
mean_ms = [r['mean_s'] * 1e3 for r in results]

plt.figure(figsize=(8, 4))
plt.plot(bytes_list, mean_ms, marker='o')
plt.xscale('log')
plt.xlabel('Payload size (bytes)')
plt.ylabel('End-to-end time (ms)')
plt.title('DMA transfer time vs payload size')
plt.grid(True)
plt.show()

In [ ]:
throughput_mb_s = [r['bytes'] / r['mean_s'] / 1e6 for r in results]

plt.figure(figsize=(8, 4))
plt.plot(bytes_list, throughput_mb_s, marker='o')
plt.xscale('log')
plt.xlabel('Payload size (bytes)')
plt.ylabel('Payload throughput (MB/s)')
plt.title('DMA throughput vs payload size')
plt.grid(True)
plt.show()

## 5. Latență vs throughput

**Latența** este dominantă când mesajele sunt mici și contează cât de repede poate fi pornită și terminată o tranzacție.

**Throughput-ul** devine relevant pentru volume mari de date și arată câte date pot fi procesate într-o unitate de timp.

Într-o aplicație reală de Edge Computing avem nevoie de ambele perspective. De exemplu, un accelerator poate avea throughput mare, dar o latență end-to-end nesatisfăcătoare dacă transferurile și sincronizările domină timpul total.


Pentru loopback, payload-ul de `N` bytes este citit din DDR și apoi scris înapoi în DDR. Dacă vrem să exprimăm traficul agregat al celor două DMA-uri, putem utiliza `2N / timp`.


In [ ]:
for r in results[-3:]:
    payload_tp = r['bytes'] / r['mean_s'] / 1e6
    aggregate_tp = 2 * r['bytes'] / r['mean_s'] / 1e6
    print(
        f"{r['bytes']:8d} B: payload={payload_tp:8.2f} MB/s, "
        f"aggregate DMA traffic={aggregate_tp:8.2f} MB/s"
    )

## 6. Exercițiu

Pe baza rezultatelor măsurate:

1. identificați transferul cu cea mai mică latență medie;
2. identificați transferul cu cel mai mare payload throughput;
3. calculați coeficientul de variație `stdev / mean` pentru fiecare dimensiune;
4. explicați în 3–5 propoziții de ce throughput-ul nu este constant pentru toate dimensiunile.


In [ ]:
# Scrieți aici analiza numerică


## 7. Întrebări

1. Care este diferența dintre latență și throughput?
2. De ce repetăm fiecare măsurare de mai multe ori?
3. Ce surse de variație introduce sistemul Linux/Python?
4. De ce performanța DMA devine importantă înainte de introducerea unui accelerator de imagine?


## 8. Concluzie

Lab3 a introdus mecanismul care va fi utilizat în laboratoarele următoare pentru transferul imaginilor între memoria DDR și acceleratoarele hardware din PL:

```text
DDR → DMA → AXI4-Stream → accelerator → AXI4-Stream → DMA → DDR
```

În Lab4 vom stabili mai întâi un **baseline software** pentru prelucrarea imaginilor pe ARM, iar ulterior îl vom compara cu accelerarea hardware.
